# Week 2: Network Properties — Assignment

**Learning objectives** — In this assignment you will:

- Rank nodes by degree centrality
- Implement betweenness centrality from scratch (Brandes' algorithm)
- Implement local clustering coefficient from scratch
- Implement closeness centrality from scratch
- Measure the correlation between centrality measures using your own implementations

## Grading

| Section | Part | Function | Points |
|---------|------|----------|--------|
| 1 | Degree Centrality | `top_k_by_degree(G, k)` | 10 |
| 2 | Betweenness | `compute_betweenness(G, normalized)` | 15 |
| 3 | Clustering | `local_clustering(G, node)` | 20 |
| 4 | Closeness | `closeness_centrality(G, node)` | 10 |
| 5 | Correlation | `centrality_correlation(G)` | 15 |
| 6 | PageRank | `compute_pagerank(G, alpha, max_iter)` | 10 |
| 7 | Assortativity | `degree_assortativity(G)` | 10 |
| — | Written Questions | — | 10 |
| | **Total** | | **100** |

## Before You Start

This assignment builds on the Week 2 lab. Make sure you are comfortable with:

- **Degree distribution** — how to read linear and log-log plots (Lab Section 2)
- **Clustering coefficient** — the formula C = 2·triangles / k(k-1) and what it measures (Lab Section 4)
- **Three centrality measures** — degree (popularity), betweenness (brokerage), closeness (proximity) and how they can disagree (Lab Section 5)
- **PageRank** — recursive importance via random walks, damping factor α (Lab Section 6)
- **Assortativity** — do hubs connect to hubs? The degree correlation coefficient r (Lab Section 7)

You will implement **all centrality measures from scratch** — do not call `nx.degree_centrality`, `nx.betweenness_centrality`, `nx.closeness_centrality`, or `nx.pagerank` in your solutions. You may use basic graph primitives (`G.degree()`, `G.neighbors()`, `nx.shortest_path_length`) as building blocks.

**Important**: In Section 5, you must **reuse your own implementations** from earlier sections rather than calling NetworkX centrality functions.

In [1]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from netsci.loaders import load_graph
from netsci.utils import SEED

In [2]:
G_karate = load_graph("karate")
G_fb = load_graph("facebook")

karate: 34 nodes, 78 edges (undirected)
facebook: 334 nodes, 2852 edges (undirected)


---
## Section 1: Degree Centrality (10 pts)

Return the top-*k* nodes by degree as a list of `(node, degree)` tuples, sorted from highest to lowest degree.

**Do not** call `nx.degree_centrality`. Use `G.degree()` to access raw degrees directly.

In [3]:
import networkx as nx

def top_k_by_degree(G, k=5):
    """Return the top-k nodes by degree.

    Do NOT call nx.degree_centrality.

    Parameters
    ----------
    G : nx.Graph
    k : int

    Returns
    -------
    list of (node, degree) tuples, sorted descending by degree.
    """
    # G.degree() returns (node, degree) pairs
    degrees = list(G.degree())
    
    # Sort the list of tuples by the degree (index 1) in descending order
    # We use a lambda function to tell Python to sort by the value, not the node ID
    sorted_degrees = sorted(degrees, key=lambda x: x[1], reverse=True)
    
    # Return only the top k elements
    return sorted_degrees[:k]

In [4]:
# --- Validation ---
_top5 = top_k_by_degree(G_karate, 5)
assert len(_top5) == 5
# Degrees should be descending
assert all(_top5[i][1] >= _top5[i + 1][1] for i in range(len(_top5) - 1))
# Node 33 or 0 should be in top 5 (they are the highest-degree nodes)
_top_nodes = {n for n, d in _top5}
assert 33 in _top_nodes or 0 in _top_nodes
print(f"Top 5 in Karate: {_top5}")
print("Section 1 passed!")

Top 5 in Karate: [(33, 17), (0, 16), (32, 12), (2, 10), (1, 9)]
Section 1 passed!


---
## Section 2: Betweenness Centrality from Scratch (15 pts)

Implement betweenness centrality **from scratch** using Brandes' algorithm. **Do not** call `nx.betweenness_centrality`.

The algorithm works as follows — for each source node *s*:

1. **Forward phase (BFS)**: Run BFS from *s*. For each node *w*, track:
   - $d[w]$: shortest distance from *s*
   - $\sigma[w]$: number of shortest paths from *s* to *w*
   - $P[w]$: list of predecessors of *w* on shortest paths from *s*
2. **Back-propagation**: Process nodes in reverse BFS order. For each node *w* with predecessor *v*:
   - $\delta[v] \mathrel{+}= \frac{\sigma[v]}{\sigma[w]} \cdot (1 + \delta[w])$
   - Then add $\delta[w]$ to $\text{betweenness}[w]$ (for $w \neq s$)
3. **Correction**: For undirected graphs, divide all values by 2 (each pair is counted twice).
4. **Normalization**: If `normalized=True`, multiply by $\frac{2}{(n-1)(n-2)}$ for undirected graphs.

You may use `collections.deque` for the BFS queue.

In [5]:
import networkx as nx

def compute_betweenness(G, normalized=True):
    """Compute betweenness centrality for all nodes from scratch (Brandes' algorithm)."""
    # Initialize centrality dict with 0.0 for all nodes
    betweenness = dict.fromkeys(G, 0.0)
    
    for s in G:
        # 1. Initialization for each source node s
        # S: Stack to store nodes in order of discovery
        # P: Predecessors on shortest paths from s
        # sigma: Number of shortest paths from s to v
        # d: Distance from s to v
        S = []
        P = {v: [] for v in G}
        sigma = dict.fromkeys(G, 0.0)
        sigma[s] = 1.0
        d = dict.fromkeys(G, -1)
        d[s] = 0
        
        # 2. BFS to find shortest paths
        queue = [s]
        while queue:
            v = queue.pop(0)
            S.append(v)
            for w in G.neighbors(v):
                # Node w found for the first time
                if d[w] < 0:
                    d[w] = d[v] + 1
                    queue.append(w)
                # Shortest path to w via v?
                if d[w] == d[v] + 1:
                    sigma[w] += sigma[v]
                    P[w].append(v)
        
        # 3. Accumulation (Backwards pass)
        delta = dict.fromkeys(G, 0.0)
        # Process nodes in reverse BFS order
        while S:
            w = S.pop()
            for v in P[w]:
                # Contribution to dependency:
                # delta_v(w) = (sigma_v / sigma_w) * (1 + delta_w)
                delta[v] += (sigma[v] / sigma[w]) * (1.0 + delta[w])
            if w != s:
                betweenness[w] += delta[w]
    
    # 4. Final step: Normalization and undirected graph adjustment
    # Brandes' algorithm counts each path twice for undirected graphs (s->t and t->s)
    # So we divide the accumulated values by 2
    for v in betweenness:
        betweenness[v] *= 0.5
        
    if normalized:
        n = len(G)
        if n > 2:
            scale = 2.0 / ((n - 1) * (n - 2))
            for v in betweenness:
                betweenness[v] *= scale
                
    return betweenness

In [6]:
# --- Validation ---
_bet = compute_betweenness(G_karate, normalized=True)
_bet_nx = nx.betweenness_centrality(G_karate, normalized=True)
assert isinstance(_bet, dict)
assert len(_bet) == G_karate.number_of_nodes()
for node in G_karate.nodes():
    assert abs(_bet[node] - _bet_nx[node]) < 1e-6, f"Mismatch at node {node}"
print("Section 2 passed!")

Section 2 passed!


---
## Section 3: Local Clustering Coefficient (20 pts)

Implement the **local clustering coefficient** for a single node **from scratch** (do not call `nx.clustering`).

Recall: for a node $v$ with degree $k_v$, the clustering coefficient is:

$$C_v = \frac{2 \times |\text{edges among neighbors of } v|}{k_v (k_v - 1)}$$

If $k_v < 2$, return 0.0 (no triangle is possible).

In [7]:
import networkx as nx

def local_clustering(G, node):
    """Compute the local clustering coefficient of a node from scratch."""
    # 1. Get the list of neighbors for the given node
    neighbors = list(G.neighbors(node))
    k = len(neighbors)
    
    # 2. Handle cases where clustering is undefined
    # If a node has fewer than 2 neighbors, it can't form a triangle.
    if k < 2:
        return 0.0
    
    # 3. Count existing edges between neighbors
    # We use a set for neighbors to ensure O(1) lookup time
    neighbor_set = set(neighbors)
    actual_edges = 0
    
    # Iterate through neighbors and check for edges between them
    for i in range(len(neighbors)):
        u = neighbors[i]
        for j in range(i + 1, len(neighbors)):
            v = neighbors[j]
            # Check if there is an edge between neighbor u and neighbor v
            if G.has_edge(u, v):
                actual_edges += 1
                
    # 4. Calculate the maximum possible edges between k neighbors
    # This is "k choose 2": k * (k - 1) / 2
    possible_edges = k * (k - 1) / 2.0
    
    return actual_edges / possible_edges

In [8]:
# --- Validation ---
for node in G_karate.nodes():
    _mine = local_clustering(G_karate, node)
    _nx_val = nx.clustering(G_karate, node)
    assert abs(_mine - _nx_val) < 1e-6, f"Node {node}: got {_mine}, expected {_nx_val}"
print("All 34 nodes match nx.clustering — Section 3 passed!")

All 34 nodes match nx.clustering — Section 3 passed!


---
## Section 4: Closeness Centrality (10 pts)

Implement **closeness centrality** for a single node **from scratch** (do not call `nx.closeness_centrality`).

$$C_C(v) = \frac{n - 1}{\sum_{u \neq v} d(v, u)}$$

where $d(v, u)$ is the shortest path length from $v$ to $u$, and $n$ is the number of nodes.

You may use `nx.shortest_path_length` to get distances.

In [9]:
import networkx as nx

def closeness_centrality(G, node):
    """Compute closeness centrality for a single node from scratch."""
    n = len(G)
    if n <= 1:
        return 0.0
    
    # 1. Calculate shortest path distances from 'node' to all other nodes
    # nx.single_source_shortest_path_length returns a dict: {target: distance}
    path_lengths = nx.single_source_shortest_path_length(G, node)
    
    # 2. Sum the distances to all reachable nodes
    # path_lengths includes the source node (distance 0), so we sum all values
    sum_distances = sum(path_lengths.values())
    
    # 3. Count reachable nodes (excluding the source itself)
    # n_reachable is the number of nodes the source can actually get to
    n_reachable = len(path_lengths) - 1
    
    if n_reachable <= 0:
        return 0.0
    
    # 4. Standard formula: (nodes_reachable / (total_nodes - 1)) * (nodes_reachable / sum_distances)
    # This accounts for disconnected components (Wasserman and Faust formula)
    # If the graph is fully connected, this simplifies to: (n-1) / sum_distances
    closeness = (n_reachable / (n - 1)) * (n_reachable / sum_distances)
    
    return float(closeness)

In [10]:
# --- Validation ---
for node in G_karate.nodes():
    _mine = closeness_centrality(G_karate, node)
    _nx_val = nx.closeness_centrality(G_karate, node)
    assert abs(_mine - _nx_val) < 1e-6, f"Node {node}: got {_mine}, expected {_nx_val}"
print("All 34 nodes match nx.closeness_centrality — Section 4 passed!")

All 34 nodes match nx.closeness_centrality — Section 4 passed!


---
## Section 5: Centrality Correlation (15 pts)

Compute the **Pearson correlation** between degree centrality and betweenness centrality across all nodes.

**Reuse your own implementations**: call your `compute_betweenness` from Section 2 and compute degree centrality as $C_D(v) = \frac{\deg(v)}{n - 1}$ (do not call `nx.degree_centrality` or `nx.betweenness_centrality`).

Use `np.corrcoef` to compute the Pearson coefficient. Return a single float.

In [11]:
import numpy as np
import networkx as nx

def centrality_correlation(G):
    """Compute Pearson r between degree and betweenness centrality."""
    nodes = list(G.nodes())
    n = len(nodes)
    
    if n <= 2:
        return 0.0 # Correlation is undefined for fewer than 3 points
        
    # 1. Compute Betweenness Centrality using your custom function
    # This returns a dict: {node: value}
    bet_dict = compute_betweenness(G, normalized=True)
    
    # 2. Compute Degree Centrality manually: deg(v) / (n - 1)
    deg_centrality = []
    bet_centrality = []
    
    for v in nodes:
        # Degree centrality calculation
        d_centrality = G.degree(v) / (n - 1)
        deg_centrality.append(d_centrality)
        
        # Betweenness centrality from our dict
        bet_centrality.append(bet_dict[v])
        
    # 3. Compute Pearson correlation using np.corrcoef
    # np.corrcoef returns a correlation matrix [[r_xx, r_xy], [r_yx, r_yy]]
    correlation_matrix = np.corrcoef(deg_centrality, bet_centrality)
    
    # The Pearson coefficient r is at index [0, 1] or [1, 0]
    r = correlation_matrix[0, 1]
    
    return float(r)

In [12]:
# --- Validation ---
_r = centrality_correlation(G_karate)
assert isinstance(_r, float)
# Degree and betweenness are positively correlated in most real networks
assert _r > 0.3, f"Expected positive correlation, got {_r}"
assert _r <= 1.0

# Verify against direct computation
_deg = nx.degree_centrality(G_karate)
_bet = nx.betweenness_centrality(G_karate)
_nodes = list(G_karate.nodes())
_d = [_deg[n] for n in _nodes]
_b = [_bet[n] for n in _nodes]
_expected_r = float(np.corrcoef(_d, _b)[0, 1])
assert abs(_r - _expected_r) < 1e-6, f"Got {_r}, expected {_expected_r}"
print(f"Pearson r (degree vs betweenness) = {_r:.4f}")
print("Section 5 passed!")

Pearson r (degree vs betweenness) = 0.9146
Section 5 passed!


---
## Section 6: PageRank from Scratch (10 pts)

Implement PageRank using the **power iteration** method. **Do not** call `nx.pagerank`.

1. Initialize: every node gets score 1/n
2. Repeat for `max_iter` iterations:
   - For each node v: new_score(v) = (1 - alpha) / n + alpha * sum(score(u) * w(u,v) / weighted_degree(u) for u in neighbors of v)
3. Return the final scores as a dictionary

Use `G.degree(u, weight="weight")` for the weighted out-degree and `G[u][v].get("weight", 1)` for edge weights (some graphs have weighted edges).

The damping factor `alpha` (default 0.85) controls how likely the random surfer is to follow a link vs jump to a random page.

In [13]:
import networkx as nx

def compute_pagerank(G, alpha=0.85, max_iter=100):
    """Compute PageRank using power iteration."""
    nodes = list(G.nodes())
    N = len(nodes)
    if N == 0:
        return {}

    # 1. Initialize PageRank scores uniformly
    pagerank = {node: 1.0 / N for node in nodes}
    
    # Pre-calculate out-degrees (weighted) to save time in the loop
    # For undirected graphs, neighbors are treated as out-edges
    out_degree = {u: G.degree(u, weight='weight') for u in nodes}

    for _ in range(max_iter):
        new_pagerank = {node: 0.0 for node in nodes}
        sink_sum = 0.0
        
        # 2. Distribute PageRank scores
        for u in nodes:
            # Handle "Dangling" nodes (sinks) with no outgoing edges
            if out_degree[u] == 0:
                sink_sum += pagerank[u]
            else:
                # Distribute score to neighbors based on edge weights
                for v in G.neighbors(u):
                    weight = G[u][v].get("weight", 1)
                    new_pagerank[v] += alpha * pagerank[u] * (weight / out_degree[u])
        
        # 3. Add the damping factor and redistribute sink/dangling scores
        # (1 - alpha) / N is the teleportation probability
        # alpha * (sink_sum / N) treats sinks as if they link to everyone
        base_score = (1.0 - alpha) / N + (alpha * sink_sum / N)
        
        for node in nodes:
            new_pagerank[node] += base_score
            
        # 4. Check for convergence (Optional, but here we run for max_iter)
        pagerank = new_pagerank

    return pagerank

In [14]:
# --- Validation ---
_pr = compute_pagerank(G_karate, alpha=0.85, max_iter=100)
_pr_nx = nx.pagerank(G_karate, alpha=0.85, max_iter=100)
assert isinstance(_pr, dict)
assert len(_pr) == G_karate.number_of_nodes()
# Check values are close to NetworkX (tolerance for convergence differences)
for node in G_karate.nodes():
    assert abs(_pr[node] - _pr_nx[node]) < 0.005, (
        f"Node {node}: got {_pr[node]:.6f}, expected {_pr_nx[node]:.6f}"
    )
# Sum should be approximately 1
assert abs(sum(_pr.values()) - 1.0) < 0.01, (
    f"Sum = {sum(_pr.values()):.4f}, expected ~1.0"
)
print(f"Top 3 by PageRank: {sorted(_pr, key=_pr.get, reverse=True)[:3]}")
print("Section 6 passed!")

Top 3 by PageRank: [33, 0, 32]
Section 6 passed!


---
## Section 7: Degree Assortativity from Scratch (10 pts)

Implement the **degree assortativity coefficient** from scratch using Pearson correlation of degrees at edge endpoints. **Do not** call `nx.degree_assortativity_coefficient`.

For each edge (u, v) in an undirected graph, include **both** orderings: (deg(u), deg(v)) and (deg(v), deg(u)). This symmetrization matches Newman's original formula. Then compute the Pearson correlation coefficient between the two degree sequences using `np.corrcoef`.

In [15]:
import numpy as np
import networkx as nx

def degree_assortativity(G):
    """Compute degree assortativity coefficient from scratch."""
    # 1. Handle edge cases (empty graphs or graphs with no edges)
    if G.number_of_edges() == 0:
        return 0.0

    # 2. Build the two degree sequences for all edges
    # We must include both (deg(u), deg(v)) and (deg(v), deg(u))
    x = []
    y = []
    
    # Pre-calculate degrees to avoid repeated lookups in the loop
    degrees = dict(G.degree())
    
    for u, v in G.edges():
        k_u = degrees[u]
        k_v = degrees[v]
        
        # Add the pair in both directions to ensure symmetry
        x.append(k_u)
        y.append(k_v)
        
        x.append(k_v)
        y.append(k_u)
        
    # 3. Compute the Pearson correlation coefficient r
    # np.corrcoef returns the correlation matrix
    if len(x) < 2:
        return 0.0
        
    correlation_matrix = np.corrcoef(x, y)
    
    # The result is the off-diagonal element
    r = correlation_matrix[0, 1]
    
    # If the denominator in Pearson is zero (e.g., a regular graph where 
    # all degrees are the same), np.corrcoef might return NaN.
    if np.isnan(r):
        return 0.0
        
    return float(r)

In [16]:
# --- Validation ---
_r = degree_assortativity(G_karate)
_r_nx = nx.degree_assortativity_coefficient(G_karate)
assert isinstance(_r, float)
assert abs(_r - _r_nx) < 0.05, f"Got {_r:.4f}, expected {_r_nx:.4f}"

# Test on Facebook too
_r_fb = degree_assortativity(G_fb)
_r_fb_nx = nx.degree_assortativity_coefficient(G_fb)
assert abs(_r_fb - _r_fb_nx) < 0.05, (
    f"Facebook: got {_r_fb:.4f}, expected {_r_fb_nx:.4f}"
)

print(f"Karate:   r = {_r:.4f} (nx: {_r_nx:.4f})")
print(f"Facebook: r = {_r_fb:.4f} (nx: {_r_fb_nx:.4f})")
print("Section 7 passed!")

Karate:   r = -0.4756 (nx: -0.4756)
Facebook: r = -0.1371 (nx: -0.1371)
Section 7 passed!


---
## Written Questions (10 pts)

### Question 1 (5 pts)

In the US power grid, what does it mean for a node to have high **betweenness centrality**?
What would happen to the network if that node (substation) failed?

*Hints to guide your thinking:*
- *Betweenness counts how many shortest paths pass **through** a node. In a sparse, tree-like network like the power grid, what happens when a node on the "trunk" is removed?*
- *Think about the difference between a node with many local connections (high degree) vs. a node that sits on the only path between two regions (high betweenness).*
- *Consider real infrastructure: if a key highway interchange fails, what happens to traffic flow?*

**Your Answer:**

A high-betweenness node acts as a critical bridge. It sits on the "shortest paths" through which power is transmitted between distant parts of the grid. It is a strategic bottleneck rather than just a busy hub.

Consequence of Failure
If this node fails, it often triggers cascading failures:

Overloading: Power is forced onto alternative paths that may not have the capacity to handle the extra load.

Fragmentation: The grid can "island" or split into disconnected pieces, leading to widespread blackouts.

Inefficiency: The average distance power must travel increases, raising transmission losses.


### Question 2 (5 pts)

Can a node have **high degree** but **low betweenness**? Describe a network structure where this happens and explain why.

*Hints to guide your thinking:*
- *Imagine a "star" subgraph: one central node connected to 50 leaf nodes, all of whom also connect to a separate hub. The central node has high degree — but do shortest paths between other nodes need to pass through it?*
- *Betweenness is about being on shortest paths **between other pairs**. A node can be popular (many friends) yet replaceable (all its friends also know each other directly).*
- *Think about the Karate Club: node 33 has the highest degree (17) — does it also have the highest betweenness? Check and explain why or why not.*

**Your Answer:**

Yes. This occurs in peripheral clusters or dense "leaf" cliques.

The Structure: A "Dandylion" or "Star-on-a-Stick"

WHY?...

High Degree: A node inside that dense cluster is connected to every other node in the same group, giving it a very high degree.

Low Betweenness: Because the cluster is tucked away at the "edge" of the network, no shortest paths between outside nodes ever pass through it. It only facilitates local traffic within its own small group, making it irrelevant for global communication across the network.